# Experiment: Case-001 Mitapivat Clinician Review Readiness Gate

Objective:
- Check that the May 17 mitapivat map converts source evidence into one clinician-review question.
- Confirm that public actions stay limited to labels, owner categories, and record domains.

Success criteria:
- seven record domains are present;
- six allowed clinician outcomes are present;
- no public question or public repo action asks for treatment, dosing, access, or eligibility decisions.


In [ ]:
# Setup: imports and source path
from __future__ import annotations

import json
from pathlib import Path

DATA_PATH = Path(
    "data/regulatory/mitapivat/2026-05-17-mitapivat-clinician-review-readiness-map.json"
)
gate = json.loads(DATA_PATH.read_text(encoding="utf-8"))
gate["gate_label"]

In [ ]:
required_domain_labels = {
    "diagnosis_genotype_context",
    "age_weight_context",
    "transfusion_burden_context",
    "iron_organ_context",
    "liver_safety_context",
    "medication_interaction_context",
    "local_access_context",
}

required_outcome_labels = {
    "review_not_ready_missing_records",
    "not_clinically_relevant_now",
    "review_with_hematologist_only",
    "refer_to_trial_or_access_owner_after_clinician_review",
    "safety_review_required_before_any_discussion",
    "lane_paused_until_pediatric_results_or_recruitment",
}

unsafe_public_terms = {
    " start ",
    " stop ",
    " dose",
    " dosing",
    " recommend",
    " eligible",
    " eligibility",
    " import",
    " travel",
}


def has_unsafe_term(text: str) -> bool:
    """Return True when public-facing text asks for unsafe action."""
    normalized = f" {text.lower()} "
    return any(term in normalized for term in unsafe_public_terms)


domains = gate["record_domains"]
outcomes = gate["allowed_clinician_outcomes"]

assert gate["gate_label"] == "mitapivat_clinician_review_readiness_ready"
assert {item["record_domain"] for item in domains} == required_domain_labels
assert {item["outcome_label"] for item in outcomes} == required_outcome_labels

for item in domains:
    assert item["why_needed"]
    assert item["public_question"]
    assert item["blocked_action"]
    assert not has_unsafe_term(item["public_question"])

for item in outcomes:
    assert item["meaning"]
    assert item["public_repo_action"]
    assert item["private_action"]
    assert not has_unsafe_term(item["public_repo_action"])

decision = {
    "gate_label": gate["gate_label"],
    "domains_checked": len(domains),
    "outcomes_checked": len(outcomes),
    "source_checks": len(gate["current_source_checks"]),
    "public_action": "single_clinician_question_no_treatment_decision",
    "clinician_question": gate["clinician_question"],
    "record_domains": [item["record_domain"] for item in domains],
    "allowed_outcomes": [item["outcome_label"] for item in outcomes],
}
decision